#### 1. Depedencies and libraries

In [107]:
!pip install -q bertopic sentence-transformers transformers torch pandas numpy matplotlib seaborn scikit-learn scipy tqdm Sastrawi

preventing kernel crash

In [108]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"]="TRUE"
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [109]:
import torch
torch.set_num_threads(1)

In [110]:
import re
import textwrap
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from collections import Counter

In [111]:
warnings.filterwarnings("ignore") # hides warning messages
sns.set_theme(style="whitegrid", context="notebook") # sets the theme for seaborn plots
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 200)


Declaring variables for config

In [112]:
csv_file = "psn_news_scraped.csv"
random_state = 42

embedding_model = "paraphrase-multilingual-MiniLM-L12-v2" # Indonesian + English
sentiment_model_id = "w11wo/indonesian-roberta-base-sentiment-classifier" # Indonesian text
sentiment_model_en = "cardiffnlp/twitter-roberta-base-sentiment" # English text

min_chars = 300 # dropping articles that's shorter than this number
min_sentences = 3 # checking if article mentions psn # of times
frequent_min_articles = 10 # minimum number for noise



#### 2. Data Loading

here loading data will be performed and to check the scrapped article or news data 

In [113]:
raw_data = pd.read_csv(csv_file)
print("Shape:", raw_data.shape)
raw_data.info()
raw_data.head(3)

Shape: (918, 7)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 918 entries, 0 to 917
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   link          918 non-null    object
 1   title         918 non-null    object
 2   snippet       918 non-null    object
 3   date          918 non-null    object
 4   source        918 non-null    object
 5   article_url   824 non-null    object
 6   article_text  764 non-null    object
dtypes: object(7)
memory usage: 50.3+ KB


,link,title,snippet,date,source,article_url,article_text
0,https://news.google.com/rss/articles/CBMia0FVX3lxTE9DQkxuemUxa0otYVFQSTBzdFo1bDN3Z1owUzNCb0gzZlMtdHdyRDVyQmY0ZDZIaDl...,Berita Kini - Menkomdigi Meutya Hafid: Satelit Nusantara Lima Bukti Nyata Kedaulatan Digital Indonesia - Kementerian...,Berita Kini - Menkomdigi Meutya Hafid: Satelit Nusantara Lima Bukti Nyata Kedaulatan Digital Indonesia Kementerian K...,"Tue, 12 May 2026 07:00:00 GMT",Kementerian Komunikasi dan Digital,https://portal.komdigi.go.id/kanal-publik/berita-kini/10210,Menteri Komunikasi dan Digital Meutya Hafid menyatakan peluncuran dan pengoperasian Satelit Nusantara Lima merupakan...
1,https://news.google.com/rss/articles/CBMikwFBVV95cUxPX2tTSjhRSnRteVoybXJpejFROThRLURodlF4OVBieVlfVGpYUXAtbUhDYVJoNmJ...,KSP Dukung Pemerataan Internet di Daerah 3T Melalui Satelit N5 - Kantor Staf Presiden,KSP Dukung Pemerataan Internet di Daerah 3T Melalui Satelit N5 Kantor Staf Presiden,"Tue, 12 May 2026 07:00:00 GMT",Kantor Staf Presiden,https://www.ksp.go.id/ksp-dukung-pemerataan-internet-di-daerah-3t-melalui-satelit-n5.html,"Kepala Staf Kepresidenan, Jenderal TNI (Purn.) Prof. Dr. H. Dudung Abdurachman, S.E., M.M., M.H., menghadiri acara P..."
2,https://news.google.com/rss/articles/CBMiswFBVV95cUxObGktUThuSjBsV2wyVDRPam92aF9fOFo4MHNWeUotQ1Q5M1pQS0Nlb0tidmlHbnd...,"Perebutan 'Kue' Internet Satelit RI Memanas, PSN Tantang Starlink - detikInet","Perebutan 'Kue' Internet Satelit RI Memanas, PSN Tantang Starlink detikInet","Wed, 13 May 2026 07:00:00 GMT",detikInet,https://inet.detik.com/telecommunication/d-8487806/perebutan-kue-internet-satelit-ri-memanas-psn-tantang-starlink,"Agus Tri Haryanto - detikInet\nDirektur Utama Pasifik Satelit Nusantara (PSN), Adi Rahman Adiwoso, mengungkap strate..."


In [114]:
raw_data.isnull().sum()

link              0
title             0
snippet           0
date              0
source            0
article_url      94
article_text    154
dtype: int64

#### 3. Data Cleaning

from data loading, it shows that there are articles left that is not talking about psn and how many nulls are there on url and text. so this section mainly adresses that along side other cleaning methods.

##### Convert Date to datetime and dropping rows without text

Parse dates and drop rows without text. for example converts date text into datetime so later, plot trends can be made. also removing rows where artricle text is missing

In [115]:
df = raw_data.copy()

df["date"] = pd.to_datetime(df["date"], format="%a, %d %b %Y %H:%M:%S GMT", errors="coerce")
print("Rows with invalid date format:", df["date"].isnull().sum())

n0 = len(df)
df = df.dropna(subset=["article_text"]).copy()
print(f"Dropped {n0 - len(df)} rows with missing article_text")
print(f"{len(df)} rows left after dropping {n0 - len(df)} rows")

Rows with invalid date format: 0
Dropped 154 rows with missing article_text
764 rows left after dropping 154 rows


##### Noise removal

news or article data scraped a bunch of text from article / news content. so the data will need to be explored first if the article contains noise, eg ads, share buttons, copyright footers, “Baca Juga”, links.

In [116]:
counter = Counter()
for text in df["article_text"]:
    lines = set(line.strip() for line in str(text).split("\n") if line.strip())
    counter.update(lines)

noise = pd.DataFrame(counter.most_common(30), columns=["line", "n_articles"])
noise = noise[noise["n_articles"] >= 3]
noise["line"] = noise["line"].str[:100]
noise

,line,n_articles
0,SCROLL TO CONTINUE WITH CONTENT,169
1,Editor:,85
2,Penulis:,75
3,Salin Pranala,75
4,Bagikan,75
5,Agus Tri Haryanto - detikInet,66
6,Baca Juga,60
7,"Dilarang keras mengambil konten, melakukan crawling atau pengindeksan otomatis untuk AI di situs web",38
8,© 2020,37
9,ADVERTISEMENT,35


In [117]:
# possible noise markers from article_text
noise_lists = [
    "scroll to continue", "advertisement", "baca juga", "bagikan", "salin pranala",
    "copyright", "©", "all rights reserved", "dilarang keras", "crawling", "izin tertulis",
    "whatsapp", "foto:", "sumber:", "read also", "also read", "http", "www.", "- detik",
    "lihat juga", "please update your browser", "mau notif", "ikuti saluran", "anda menyukai artikel",
]

In [118]:
def show_noise(idx):
    row = df.loc[idx]
    print(row["title"], "|", row["source"])
    print()

    # 1) split the article into lines and drop the empty ones
    lines = []
    for line in str(row["article_text"]).split("\n"):
        if line.strip() != "":
            lines.append(line.strip())

    # 2) check every line for noise words
    for number, line in enumerate(lines, 1):
        is_noise = False
        for word in noise_lists:
            if word in line.lower():
                is_noise = True

        if is_noise:
            print(f"[Possible Noise] line {number}: {line}")
        else:
            print(f"[keep]   line {number}: {line}")

In [119]:
show_noise(idx=df.sample(1).index[0])      # a random article; run again for another one
# show_noise(keyword="SATRIA-1")   

BTS 4G Bakti Diresmikan Jokowi 28 Desember, Bareng Satria-1? - detikInet | detikInet

[Possible Noise] line 1: Adi Fida Rahman - detikInet
[keep]   line 2: Setelah sempat terganjal kasus korupsi, pembangunan BTS 4G yang garap Badan Aksesibilitas Telekomunikasi dan Informasi (Bakti) rampung. Pada 28 Desember, BTS 4G bakal diresmikan pengoperasiannya langsung oleh Presiden Joko Widodo.
[keep]   line 3: "BTS 4G diresmikan pengoperasiannya oleh bapak presiden Jokowi di Talaud, Sulawesi Utara," ujad Budi Arie Setiadi , Menteri Komunikasi dan Informatika (Menkominfo), usai konferensi pers Surat Edaran Etika Pengguna AI di Kantor Kominfo, Jumat (22/12023).
[keep]   line 4: Budi sempat ditanya apakah peresmian BTS 4G ini akan berbarengan dengan dimulainya pengoperasian Satelit Satria-1. Namun dia tidak memberikan jawaban yang pasti.
[Possible Noise] line 5: SCROLL TO CONTINUE WITH CONTENT
[keep]   line 6: "Jadi pokoknya presiden mengoperasikan peresmian BTS 4G Bakti tanggal 28 Desember 2023 di

based on the result, it shows that there are a some of noise on most articles. hence it need to be removed 

In [120]:
noise_patterns = [
    r"^SCROLL TO CONTINUE", r"^advertisement$", r"^baca\s*juga", r"^bagikan$", r"^salin pranala$",
    r"^penulis\s*:", r"^editor\s*:", r"^translator\s*:", r"^pewarta", r"^can't find what",
    r"^available on", r"^ikuti saluran", r"^mau notif", r"^dapatkan aplikasi", r"tanpa suara beda",
    r"^©", r"copyright", r"all rights reserved", r"dilarang keras", r"crawling atau pengindeksan",
    r"izin tertulis dari direksi", r"^alamat\s*:", r"whatsapp\s*:", r"^address\s*:",
    r"-\s*detik\w+$",          # bylines / photo credits such as "Agus Tri Haryanto - detikInet"
    r"^foto\s*:", r"^sumber\s*:", r"^source\s*:", r"^read also", r"^also read",
]

noise_re = re.compile("|".join(noise_patterns), re.IGNORECASE)
url_re = re.compile(r"https?://\S+|www\.\S+")

In [121]:
def clean_article(text: str) -> str:
    lines = []
    for line in str(text).replace("\xa0", " ").split("\n"):
        line = line.strip()
        if not line or any(noise_word in line.lower() for noise_word in noise_lists):
            continue
        line = url_re.sub("", line)
        line = line.replace("[…]", "").replace("[...]", "")
        line = re.sub(r"\s+", " ", line).strip()
        if line:
            lines.append(line)
    return " ".join(lines)

based on the scrapped data, on every headline there is also publisher name for example ``` - detik.com```, some are duplicates. hence it needs to be removed.

In [122]:
def strip_source(title, source):
    title = str(title).strip()
    base = lambda x: re.sub(r"\.(com|co\.id|id|net|org)$", "", x.strip().lower())
    src = base(str(source))
    while True:                                   # the publisher name can appear twice
        m = re.search(r"\s+-\s+([^-]+)$", title)
        if not m:
            break
        suffix = base(m.group(1))
        if suffix == src or (len(suffix) > 3 and (suffix in src or src in suffix)):
            title = title[:m.start()].strip()
        else:
            break
    return title

In [123]:
df["text"] = df["article_text"].apply(clean_article)
df["title_clean"] = [strip_source(t, s) for t, s in zip(df["title"], df["source"])]

ex = df.sample(1, random_state=1).iloc[0]
print("BEFORE:", ex["article_text"][:400].replace("\n", " | "))
print()
print("AFTER :", ex["text"][:400])
print()
print("Title before:", ex["title"])
print("Title after :", ex["title_clean"])

BEFORE: Agus Tri Haryanto - detikInet | Dalam hitungan tiga hari lagi, Indonesia akan memiliki satelit internet pertama. Satelit Republik Indonesia atau Satria-1 dijadwalkan meluncur dari Cape Canaveral, Florida, Amerika Serikat, pada 18 Juni 2023 waktu setempat atau 19 Juni 2023 waktu Indonesia. | Sebelumnya satelit ini dirakit oleh Thales Alenia Space di Prancis sejak September 2020 hingga Mei 2023. Rencana

AFTER : Dalam hitungan tiga hari lagi, Indonesia akan memiliki satelit internet pertama. Satelit Republik Indonesia atau Satria-1 dijadwalkan meluncur dari Cape Canaveral, Florida, Amerika Serikat, pada 18 Juni 2023 waktu setempat atau 19 Juni 2023 waktu Indonesia. Sebelumnya satelit ini dirakit oleh Thales Alenia Space di Prancis sejak September 2020 hingga Mei 2023. Rencana awal, akan diangkut dari Pran

Title before: H-3 Peluncuran Satelit Satria-1, Sejarah Indonesia Akan Tercipta - detikInet
Title after : H-3 Peluncuran Satelit Satria-1, Sejarah Indonesia Akan Tercipta


the data should be cleaned after the noise removal. below i will do a quick check if its already cleaned 

In [124]:
lines_before = df["article_text"].astype(str).apply(lambda t: len([l for l in t.split("\n") if l.strip()])).sum()
print("Lines before cleaning:", lines_before)
print("Characters before:", int(df["article_text"].str.len().sum()), "-> after:", int(df["text"].str.len().sum()))

Lines before cleaning: 14798
Characters before: 3039080 -> after: 2836403


In [125]:
df.head(3)

,link,title,snippet,date,source,article_url,article_text,text,title_clean
0,https://news.google.com/rss/articles/CBMia0FVX3lxTE9DQkxuemUxa0otYVFQSTBzdFo1bDN3Z1owUzNCb0gzZlMtdHdyRDVyQmY0ZDZIaDl...,Berita Kini - Menkomdigi Meutya Hafid: Satelit Nusantara Lima Bukti Nyata Kedaulatan Digital Indonesia - Kementerian...,Berita Kini - Menkomdigi Meutya Hafid: Satelit Nusantara Lima Bukti Nyata Kedaulatan Digital Indonesia Kementerian K...,2026-05-12 07:00:00,Kementerian Komunikasi dan Digital,https://portal.komdigi.go.id/kanal-publik/berita-kini/10210,Menteri Komunikasi dan Digital Meutya Hafid menyatakan peluncuran dan pengoperasian Satelit Nusantara Lima merupakan...,Menteri Komunikasi dan Digital Meutya Hafid menyatakan peluncuran dan pengoperasian Satelit Nusantara Lima merupakan...,Berita Kini - Menkomdigi Meutya Hafid: Satelit Nusantara Lima Bukti Nyata Kedaulatan Digital Indonesia
1,https://news.google.com/rss/articles/CBMikwFBVV95cUxPX2tTSjhRSnRteVoybXJpejFROThRLURodlF4OVBieVlfVGpYUXAtbUhDYVJoNmJ...,KSP Dukung Pemerataan Internet di Daerah 3T Melalui Satelit N5 - Kantor Staf Presiden,KSP Dukung Pemerataan Internet di Daerah 3T Melalui Satelit N5 Kantor Staf Presiden,2026-05-12 07:00:00,Kantor Staf Presiden,https://www.ksp.go.id/ksp-dukung-pemerataan-internet-di-daerah-3t-melalui-satelit-n5.html,"Kepala Staf Kepresidenan, Jenderal TNI (Purn.) Prof. Dr. H. Dudung Abdurachman, S.E., M.M., M.H., menghadiri acara P...","Kepala Staf Kepresidenan, Jenderal TNI (Purn.) Prof. Dr. H. Dudung Abdurachman, S.E., M.M., M.H., menghadiri acara P...",KSP Dukung Pemerataan Internet di Daerah 3T Melalui Satelit N5
2,https://news.google.com/rss/articles/CBMiswFBVV95cUxObGktUThuSjBsV2wyVDRPam92aF9fOFo4MHNWeUotQ1Q5M1pQS0Nlb0tidmlHbnd...,"Perebutan 'Kue' Internet Satelit RI Memanas, PSN Tantang Starlink - detikInet","Perebutan 'Kue' Internet Satelit RI Memanas, PSN Tantang Starlink detikInet",2026-05-13 07:00:00,detikInet,https://inet.detik.com/telecommunication/d-8487806/perebutan-kue-internet-satelit-ri-memanas-psn-tantang-starlink,"Agus Tri Haryanto - detikInet\nDirektur Utama Pasifik Satelit Nusantara (PSN), Adi Rahman Adiwoso, mengungkap strate...","Direktur Utama Pasifik Satelit Nusantara (PSN), Adi Rahman Adiwoso, mengungkap strategi perusahaan dalam menghadapi ...","Perebutan 'Kue' Internet Satelit RI Memanas, PSN Tantang Starlink"


##### Dropping short articles

some pages returned only short texts, eg copyright footer, " read more ", or a cookie warnings. so those short texts will be dropped as well

In [126]:
n0 = len(df)
df["n_chars"] = df["text"].str.len()
df = df[df["n_chars"] >= min_chars].copy()
print(f"Dropped {n0 - len(df)} rows with text shorter than {min_chars} characters")

Dropped 93 rows with text shorter than 300 characters


##### Removing duplicates

google news often returns the same article from different links, so those dudplicates will be removed.

In [127]:
n0 = len(df)
df = df.drop_duplicates(subset="text").drop_duplicates(subset="title_clean").copy()
print(f"Dropped {n0 - len(df)} duplicate rows based on text and title_clean")

Dropped 2 duplicate rows based on text and title_clean


##### PSN titles only

from the result of scrapping, there are some articles that are not about psn that has been scraped. so it needs to be removed. the article will be kept only if psn is in the title or psn mentioned at least twice in the content.

In [128]:
psn_re = re.compile(
    r"\bPSN\b|Pasifik Satelit|Satelit Nusantara|Nusantara\s+(Satu|Dua|Tiga|Lima|One|Two|Three|Five)",
    flags=re.IGNORECASE,
)

def count_psn_sentences(text):
    sentences = {s.strip() for s in re.split(r"(?<=[.!?])\s+", text)}   # a set removes repeated sentences
    return sum(bool(psn_re.search(s)) for s in sentences)

n0 = len(df)
df["psn_sentences"] = df["text"].apply(count_psn_sentences)
df["psn_in_title"] = df["title_clean"].apply(lambda t: bool(psn_re.search(t)))

df = df[df["psn_in_title"] | (df["psn_sentences"] >= min_sentences)].copy()
log_step("not really about PSN", n0, len(df))

not really about PSN: dropped 415 -> 254 left


##### Manual Review

in total there are 254 article left after removing articles that are not about psn. here i need to make sure if the rest of the articles are about psn by manually reviewing some of the data left.

In [134]:
borderline = df[~df["psn_in_title"] & (df["psn_sentences"] <= 4)]

print(f"{len(borderline)} borderline articles to review")
borderline[["title_clean", "source", "psn_sentences"]].sort_values("psn_sentences").reset_index(drop=True)

44 borderline articles to review


,title_clean,source,psn_sentences
0,Satelit Nano Pertama Buatan Mahasiswa Indonesia Siap Meluncur,Liputan6.com,3
1,"Tahun Depan, Indonesia Akan Luncurkan Tiga Satelit",MMC Kotawaringin Barat,3
2,"Mengenal Satelit SATRIA-1, Segera Diluncurkan di Tengah Kasus BAKTI",CNN Indonesia,3
3,Penampakan Satelit Satria-1 yang Siap Meluncur Pakai Roket SpaceX,detikFinance,3
4,Cara Satelit Satria-1 Bekerja Sebar Internet di Pelosok RI,detikInet,3
5,Saat Satelit Elon Musk Bakal Jualan Eceran di Tanah Air,Kompas.id,3
6,"Satelit SATRIA-1 Berhasil Ngorbit, Bersiap Kasih Internet Gratis",CNN Indonesia,3
7,"Adi Rahman, Penemu & Pengusaha yang Bakal Luncurkan Satelit Terbesar se-Asia",kumparan,3
8,Alhamdulillah! Satelit Satria-1 Sampai Slot Orbit 146 Derajat Bujur Timur,detikInet,3
9,"Thales Alenia Serahkan Satelit Satria-1 ke SNT, Siap Layani 20.000 Titik",Bisnis Tekno,3


##### Distribution of languages for articles

since the data is mixed between Indonesian and English, we need to know the distribution of languages to pick the right model.

In [129]:
ID = re.compile(r"\b(yang|dan|di|untuk|dengan|dari|pada|ini|itu|dalam|akan)\b", re.IGNORECASE)
EN = re.compile(r"\b(the|and|of|to|in|for|with|that|is|are)\b", re.IGNORECASE)

def detect_lang(text: str) -> str:
    sample = text[:3000]
    return "en" if len(EN.findall(sample)) > len (ID.findall(sample)) else "id"

df["lang"] = df["text"].apply(detect_lang)
df["lang"].value_counts()

lang
id    216
en     38
Name: count, dtype: int64

##### Helper columns and reset index

After cleaning, adding helper columns (`year`, `year_month`, `n_words`) for EDA charts, sort the articles by date, and reset the index so the rows doesnt have empty gaps. `doc_id` column gives each article a unique ID, which is used to match topic and sentiment results to the right article.

In [133]:
df["year"] = df["date"].dt.year
df["year_month"] = df["date"].dt.to_period("M").dt.to_timestamp()
df["n_words"] = df["text"].str.split().str.len()

df = df.sort_values("date").reset_index(drop=True)
df["doc_id"] = df.index

print("Cleaned dataset:", df.shape)
df[["doc_id", "date", "source", "lang", "title_clean", "n_words"]].head(10)

Cleaned dataset: (254, 17)


,doc_id,date,source,lang,title_clean,n_words
0,0,2017-05-17 07:00:00,Liputan6.com,id,Satelit Nano Pertama Buatan Mahasiswa Indonesia Siap Meluncur,283
1,1,2017-12-05 08:00:00,kumparan,id,PSN Bikin Satelit Baru untuk Sebar Internet di Indonesia,538
2,2,2019-01-23 08:00:00,Bisnis Tekno,id,Satelit Kapasitas Jumbo Nusantara Satu Mengorbit pada Februari,258
3,3,2019-01-23 08:00:00,Suara.com,id,PSN Siap Luncurkan Satelit dengan Kapasitas Broadband Terbesar di Indonesia,317
4,4,2019-01-23 08:00:00,kumparan,id,"Satelit Nusantara Satu PSN Meluncur Februari 2019, Dibawa oleh SpaceX",399
5,5,2019-01-25 08:00:00,ANTARA News Jateng,id,Satelit Nusantara Satu buatan Indonesia segera meluncur,320
6,6,2019-02-14 08:00:00,Majalah ICT,id,Satelit Palapa Nusantara Dua Indosat Ooredoo Ditargetkan akan Luncur Mei 2020,433
7,7,2019-02-19 08:00:00,Spaceflight Now,en,Falcon 9 launch set for Thursday at Cape Canaveral,1112
8,8,2019-02-21 08:00:00,Space,en,SpaceX Rocket Launches 1st Private Moon Lander for Israel,1345
9,9,2019-02-21 08:00:00,Kumparan.com,id,"Nusantara Satu, Satelit Broadband yang Merajut Indonesia",336


##### Cleaning Summary and Saving Cleaning Results


the cleaned data will be 

In [137]:
print("Missing text:", df["text"].isnull().sum())
print("Missing date:", df["date"].isnull().sum())
print("Articles:", len(df), "| date range:", df["date"].min().date(), "->", df["date"].max().date())
print()
print(df["year"].value_counts().sort_index())

Missing text: 0
Missing date: 0
Articles: 254 | date range: 2017-05-17 -> 2026-07-09

year
2017     2
2019    39
2020    27
2021    13
2022     4
2023    44
2024    11
2025    82
2026    32
Name: count, dtype: int64


In [138]:
clean_cols = ["doc_id", "date", "year", "year_month", "source", "lang",
              "title", "title_clean", "article_url", "text", "n_words", "psn_sentences"]

df[clean_cols].to_csv("psn_article_cleaned.csv", index=False)
print(f"Saved {len(df)} cleaned articles to psn_article_cleaned.csv")

Saved 254 cleaned articles to psn_article_cleaned.csv


In [139]:
pd.read_csv("psn_article_cleaned.csv").head(10)

,doc_id,date,year,year_month,source,lang,title,title_clean,article_url,text,n_words,psn_sentences
0,0,2017-05-17 07:00:00,2017,2017-05-01,Liputan6.com,id,Satelit Nano Pertama Buatan Mahasiswa Indonesia Siap Meluncur - Liputan6.com,Satelit Nano Pertama Buatan Mahasiswa Indonesia Siap Meluncur,https://www.liputan6.com/tekno/read/2954873/satelit-nano-pertama-buatan-mahasiswa-indonesia-siap-meluncur,"Rencananya, satelit ini akan mulai meluncur pada akhir 2017 Liputan6.com, Jakarta - Sekelompok mahasiswa Surya Unive...",283,3
1,1,2017-12-05 08:00:00,2017,2017-12-01,kumparan,id,PSN Bikin Satelit Baru untuk Sebar Internet di Indonesia - kumparan.com - kumparan,PSN Bikin Satelit Baru untuk Sebar Internet di Indonesia,https://kumparan.com/kumparantech/psn-bikin-satelit-baru-untuk-sebar-internet-di-indonesia-27431110790324244,"kumparanTECH 5 Desember 2017 18:07 WIB Perusahaan jasa telekomunikasi dan informasi berbasis satelit, PT Pasifik Sat...",538,13
2,2,2019-01-23 08:00:00,2019,2019-01-01,Bisnis Tekno,id,Satelit Kapasitas Jumbo Nusantara Satu Mengorbit pada Februari - Bisnis Tekno,Satelit Kapasitas Jumbo Nusantara Satu Mengorbit pada Februari,https://teknologi.bisnis.com/read/20190123/101/881661/satelit-kapasitas-jumbo-nusantara-satu-mengorbit-pada-februari,"Bisnis.com, JAKARTA — PT Pasifik Satelit Nusantara (PSN) akan meluncurkan satelit terbaru yang diberi nama Nusantara...",258,7
3,3,2019-01-23 08:00:00,2019,2019-01-01,Suara.com,id,PSN Siap Luncurkan Satelit dengan Kapasitas Broadband Terbesar di Indonesia - Suara.com,PSN Siap Luncurkan Satelit dengan Kapasitas Broadband Terbesar di Indonesia,https://www.suara.com/tekno/2019/01/23/150541/psn-siap-luncurkan-satelit-dengan-kapasitas-broadband-terbesar-di-indo...,Available On Dythia Novianty Tivan Rahmat Suara.com - PT. Pasifik Satelit Nusantara ( PSN ) segera meluncurkan Satel...,317,7
4,4,2019-01-23 08:00:00,2019,2019-01-01,kumparan,id,"Satelit Nusantara Satu PSN Meluncur Februari 2019, Dibawa oleh SpaceX - kumparan.com - kumparan","Satelit Nusantara Satu PSN Meluncur Februari 2019, Dibawa oleh SpaceX",https://kumparan.com/kumparantech/satelit-nusantara-satu-psn-meluncur-februari-2019-dibawa-oleh-spacex-1548229076280...,kumparanTECH 23 Januari 2019 14:46 WIB Rencana PT Pasifik Satelit Nusantara ( PSN ) meluncurkan satelit baru untuk I...,399,14
5,5,2019-01-25 08:00:00,2019,2019-01-01,ANTARA News Jateng,id,Satelit Nusantara Satu buatan Indonesia segera meluncur - ANTARA News Jateng,Satelit Nusantara Satu buatan Indonesia segera meluncur,https://jateng.antaranews.com/berita/211254/satelit-nusantara-satu-buatan-indonesia-segera-meluncur,Jakarta (Antaranews Jateng) - Perusahaan telekomunikasi PT Pasifik Satelit Nusantara akan meluncurkan satelit Nusant...,320,9
6,6,2019-02-14 08:00:00,2019,2019-02-01,Majalah ICT,id,Satelit Palapa Nusantara Dua Indosat Ooredoo Ditargetkan akan Luncur Mei 2020 - Majalah ICT,Satelit Palapa Nusantara Dua Indosat Ooredoo Ditargetkan akan Luncur Mei 2020,https://www.majalahict.com/satelit-palapa-nusantara-dua-indosat-ooredoo-ditargetkan-akan-luncur-mei-2020/,"Kamis, Februari 14 7:13 Redaksi Berita Komentar Dinonaktifkan pada Satelit Palapa Nusantara Dua Indosat Ooredoo Dita...",433,4
7,7,2019-02-19 08:00:00,2019,2019-02-01,Spaceflight Now,en,Falcon 9 launch set for Thursday at Cape Canaveral - Spaceflight Now,Falcon 9 launch set for Thursday at Cape Canaveral,https://spaceflightnow.com/2019/02/19/falcon-9-launch-set-for-thursday-at-cape-canaveral/,File photo of a Falcon rocket inside SpaceX's hangar at pad 40. Credit: SpaceX SpaceX is set to end a two-month laun...,1112,8
8,8,2019-02-21 08:00:00,2019,2019-02-01,Space,en,SpaceX Rocket Launches 1st Private Moon Lander for Israel - Space,SpaceX Rocket Launches 1st Private Moon Lander for Israel,https://www.space.com/spacex-israeli-moon-lander-satellites-launch-success.html,"When you purchase through links on our site, we may earn an affiliate commission. Here’s how it works . CAPE CANAVER...",1345,5
9,9,2019-02

#### 4. EDA